# Unit 11 — Diffusion training objective

Training adds known noise to clean action windows. The network learns to predict that noise. The padded action slots are copied values, so the loss must ignore them.

**Your result:** the functions that train a diffusion policy: cumulative alphas, forward noising, timestep selection, a masked noise loss, and a loss callback that the training loop uses to update a real denoiser. For the same random numbers your loss equals the pinned LeRobot loss.


In [ ]:
import sys
import time as wall_clock
from collections.abc import Callable, Mapping
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn

WORKSPACE = next(
    p
    for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "manifests/act_references.yaml").is_file()
)
sys.path.insert(0, str(WORKSPACE / "src"))
import signal
from types import FrameType

import ipywidgets as widgets
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from IPython.display import display
from lerobot.policies.diffusion import DiffusionPolicy
from torch.utils.data import default_collate

from vla_lab.evaluation.diffusion import evaluate_diffusion_reference
from vla_lab.teaching.act_model import open_libero_dataset
from vla_lab.teaching.diffusion import (
    DDPMSchedule,
    LeRobotDenoiser,
    cosine_betas,
    load_prepared_diffusion,
)
from vla_lab.teaching.interfaces import DiffusionTrainingModel
from vla_lab.training.diffusion import (
    diffusion_delta_timestamps,
    load_diffusion_bundle,
    train_diffusion_reference,
)

In [ ]:
# Supplied: a helper that the checks use
def close(actual: torch.Tensor, expected: torch.Tensor, message: str, atol: float = 1e-5) -> None:
    """Compare two tensors and say what failed."""
    assert actual.shape == expected.shape, (
        f"{message}: shape {tuple(actual.shape)}, expected {tuple(expected.shape)}"
    )
    worst = (actual.detach() - expected.detach()).abs().max().item()
    assert worst <= atol, f"{message}: largest difference {worst:.2e} is above {atol:.0e}"

## Background: the forward process and what the network learns

Start from a clean action window $x_0$ (16 steps × 7 values, scaled to $[-1, 1]$). The **forward process** adds Gaussian noise in $T=100$ small steps. A closed form jumps straight to step $t$:

$$x_t=\sqrt{\bar{\alpha}_t}\,x_0+\sqrt{1-\bar{\alpha}_t}\,\varepsilon, \qquad \varepsilon\sim\mathcal{N}(0,I), \qquad \bar{\alpha}_t=\prod_{s=0}^{t}\left(1-\beta_s\right)$$

Where:
* $x_0$ - the clean action window, shape $(H,A)$ with $H=16$ steps and $A=7$ values.
* $t$ - the noise step, an integer from $0$ to $T-1$ (the code counts from 0; the DDPM paper counts from 1).
* $T$ - the number of noise steps, here 100.
* $\beta_s$ - the noise variance added at step $s$, from the fixed beta schedule.
* $\bar{\alpha}_t$ - the fraction of signal variance left after step $t$.
* $\varepsilon$ - standard normal noise with the shape of $x_0$.
* $x_t$ - the noisy window at step $t$.

Small $t$ is almost clean; large $t$ is almost pure noise. The **network** $\varepsilon_\theta(x_t, t, o)$, with parameters $\theta$ and observation $o$, is trained to predict the noise $\varepsilon$ that was added. The loss is the mean squared error over *real* action steps. Sampling later runs the process in reverse (Unit 14).

The pinned recipe, from `lerobot/policies/diffusion` at revision `e0d50211ef236143ae867228662b7dfaba554f02`: `T = 100`, the `squaredcos_cap_v2` (cosine) beta schedule, noise prediction, padded steps masked (`do_mask_loss_for_padding=True`), observation history 2, horizon 16, 8 executed actions. The current trainer uses Adam with a cosine learning-rate schedule and **no EMA** (exponential moving average of weights). Sources: [DDPM](https://arxiv.org/abs/2006.11239) sections 2 and 3, equations 4 and 12, Algorithm 1; [Diffusion Policy](https://diffusion-policy.cs.columbia.edu/).

**Supplied here:** the beta schedule, the denoiser network (supplied here; Units 12 and 13 open its internals), the data reader and the real training loop. **You write** the five functions that turn a clean window into a training signal.
### Worked example: one noising step

Take two beta values $\beta=[0.1,\,0.2]$. Then $\bar{\alpha}=[0.9,\,0.72]$. For one scalar action $x_0=1$, noise $\varepsilon=-1$, and timestep $t=1$, the noised value is $\sqrt{0.72}\cdot1+\sqrt{0.28}\cdot(-1)\approx0.32$. The same `t` applies to every time/action value in that training row; another row can use another timestep. If the final slot is padding, its loss mask is zero, so it adds no gradient.


## Exercise 11.1 — Cumulative alphas

**Purpose.** The noise level at step $t$ depends on the product of all $(1-\beta_s)$ up to $t$. Compute it once for all steps.

**Contract (shapes, units, masks).**

- `cumulative_alphas(betas)`: `betas` is a `float32` tensor `(T,)` with values in `(0, 1)`.
- Return `alpha_bar` `(T,)`, `float32`, with `alpha_bar[t]` $=\bar{\alpha}_t=\prod_{s=0}^{t}(1-\beta_s)$, where $\beta_s$ is `betas[s]` (the product includes step $t$). So `alpha_bar[0] = 1 - betas[0]`.
- Do not loop in Python over steps; one tensor operation is enough.

**Method and source.** [DDPM](https://arxiv.org/abs/2006.11239), section 2, just before equation 4, with indices from 0:

$$\alpha_t=1-\beta_t, \qquad \bar{\alpha}_t=\prod_{s=0}^{t}\alpha_s$$

Where $\beta_t$ is `betas[t]` and $\bar{\alpha}_t$ is `alpha_bar[t]`.

**Pointers.** Turn betas into alphas first. Then use a cumulative product along the only axis.

<details><summary>Detailed hint</summary>

`torch.cumprod(1.0 - betas, dim=0)`.

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_cumulative_alphas` in [lesson11_checks.py](lesson11_checks.py).

In [ ]:
# Reference solution for Exercise 11.1
def cumulative_alphas(betas: torch.Tensor) -> torch.Tensor:
    """``alpha_bar[t] = prod_{s <= t} (1 - beta[s])`` (DDPM paper, equation 4). Shape ``(T,)``."""
    return torch.cumprod(1.0 - betas, dim=0)

In [ ]:
# Check: Exercise 11.1
betas = cosine_betas(100)
alpha_bar = cumulative_alphas(betas)
assert alpha_bar.shape == (100,) and alpha_bar.dtype == torch.float32, (
    "Expected a float32 tensor of shape (T,)."
)
close(alpha_bar[0], 1 - betas[0], "alpha_bar[0] is 1 - beta[0]")
close(
    alpha_bar[2],
    (1 - betas[0]) * (1 - betas[1]) * (1 - betas[2]),
    "alpha_bar[2] is the product of three alphas",
)
assert (alpha_bar[1:] < alpha_bar[:-1]).all(), "alpha_bar must decrease with t."
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
close(alpha_bar, scheduler.alphas_cumprod, "differs from diffusers' alphas_cumprod", atol=1e-7)
close(
    cumulative_alphas(torch.tensor([0.5, 0.5, 0.5])),
    torch.tensor([0.5, 0.25, 0.125]),
    "constant betas",
    atol=1e-7,
)
print("Exercise 11.1 passed")

# More behaviour checks: lesson11_checks.py, next to this notebook.
from lesson11_checks import check_cumulative_alphas

check_cumulative_alphas(cumulative_alphas)

## Exercise 11.2 — Forward noising

**Purpose.** Jump from a clean action window to its noisy version at any step in one operation.

**Contract (shapes, units, masks).**

- `x0` and `noise`: `(B, H, A)`. `timesteps`: `(B,)` `long`, values in `[0, T)`. `alpha_bar`: `(T,)`.
- Each batch row uses its *own* timestep.
- Return `x_t` `(B, H, A)` with $x_t=\sqrt{\bar{\alpha}_t}\,x_0+\sqrt{1-\bar{\alpha}_t}\,\varepsilon$, where for row $b$, $\bar{\alpha}_t$ is `alpha_bar[timesteps[b]]`, $x_0$ is `x0[b]`, and $\varepsilon$ is `noise[b]`.
- Do not change `x0` or `noise` in place. Gradients must flow to `x0` and `noise`.

**Method and source.** [DDPM](https://arxiv.org/abs/2006.11239), equation 4:

$$q(x_t\mid x_0)=\mathcal{N}\!\left(\sqrt{\bar{\alpha}_t}\,x_0,\ (1-\bar{\alpha}_t)\,I\right)$$

Where $q(x_t\mid x_0)$ is the distribution of the noisy window given the clean one, and $\bar{\alpha}_t$ is `alpha_bar[t]` for that row's timestep. Sampling from it gives the formula in the contract. This is also what `DDPMScheduler.add_noise` computes.

**Pointers.** Look up the per-row coefficients with the timestep tensor. They have shape `(B,)`; reshape them so they broadcast against `(B, H, A)`.

<details><summary>Detailed hint</summary>

```python
abar = alpha_bar.to(device=x0.device, dtype=x0.dtype)[timesteps]       # (B,)
abar = abar.reshape(-1, 1, 1)                                          # (B, 1, 1)
return abar.sqrt() * x0 + (1.0 - abar).sqrt() * noise
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_forward_noise` in [lesson11_checks.py](lesson11_checks.py).

In [ ]:
# Reference solution for Exercise 11.2
def forward_noise(
    x0: torch.Tensor, noise: torch.Tensor, timesteps: torch.Tensor, alpha_bar: torch.Tensor
) -> torch.Tensor:
    """Noise a clean sample in one step (DDPM equation 4).

    ``x_t = sqrt(alpha_bar[t]) * x0 + sqrt(1 - alpha_bar[t]) * noise``, with ``t`` chosen per
    batch row. ``x0`` and ``noise`` are ``(B, H, A)``, ``timesteps`` is ``(B,)``.
    """
    abar = alpha_bar.to(device=x0.device, dtype=x0.dtype)[timesteps].reshape(
        -1, *([1] * (x0.ndim - 1))
    )
    return abar.sqrt() * x0 + (1.0 - abar).sqrt() * noise

In [ ]:
# Check: Exercise 11.2
scheduler = DDPMScheduler(num_train_timesteps=100, beta_schedule="squaredcos_cap_v2")
alpha_bar = (1.0 - cosine_betas(100)).cumprod(0)
x0, noise = torch.randn(4, 16, 7, requires_grad=True), torch.randn(4, 16, 7)
t = torch.tensor([0, 17, 50, 99])
out = forward_noise(x0, noise, t, alpha_bar)
assert out.shape == x0.shape, "Return the same shape as x0."
close(out, scheduler.add_noise(x0.detach(), noise, t), "x_t differs from diffusers' add_noise")
row = forward_noise(x0[1:2].detach(), noise[1:2], torch.tensor([17]), alpha_bar)
close(out[1:2], row, "a row must depend only on its own timestep")
low = forward_noise(x0.detach(), noise, torch.zeros(4, dtype=torch.long), alpha_bar)
assert (low - x0.detach()).abs().max() < 0.25, "At t = 0 the sample is almost the clean window."
out.sum().backward()
close(
    x0.grad[0], alpha_bar[0].sqrt().expand(16, 7), "d x_t / d x0 is sqrt(alpha_bar[t])", atol=1e-6
)
print("Exercise 11.2 passed")

# More behaviour checks: lesson11_checks.py, next to this notebook.
from lesson11_checks import check_forward_noise

check_forward_noise(forward_noise)

## Exercise 11.3 — Choose noise and timesteps

**Purpose.** Each training example needs its own noise and its own random noise level. Draw both in a fixed order so a run can be repeated exactly.

**Contract (shapes, units, masks).**

- `actions (B, H, A)` float tensor, `num_train_timesteps` (int `T`), `generator` (a `torch.Generator` on the same device as `actions`, or `None` for the global generator).
- Draw **first** `noise ~ N(0, I)` with the shape, dtype and device of `actions`; **then** `timesteps`, `B` integers drawn uniformly from `0 … T-1` (`T` itself is excluded), dtype `long`, on the same device.
- Use the same `generator` for both draws. Return `(noise, timesteps)`.

**Method and source.** Algorithm 1 in [DDPM](https://arxiv.org/abs/2006.11239): $t\sim\operatorname{Uniform}(\{1,\dots,T\})$ and $\varepsilon\sim\mathcal{N}(0,I)$. Indices here start at 0, so $t\in\{0,\dots,T-1\}$. The pinned `compute_loss` draws noise first, then `torch.randint`.

**Pointers.** Two calls: `torch.randn` with an explicit shape, `torch.randint` with `low`, `high` and a size. Pass the generator to both.

<details><summary>Detailed hint</summary>

```python
noise = torch.randn(actions.shape, generator=generator, device=actions.device, dtype=actions.dtype)
timesteps = torch.randint(0, num_train_timesteps, (actions.shape[0],), generator=generator, device=actions.device)
return noise, timesteps
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_sample_noise_and_timesteps` in [lesson11_checks.py](lesson11_checks.py).

In [ ]:
# Reference solution for Exercise 11.3
def sample_noise_and_timesteps(
    actions: torch.Tensor, num_train_timesteps: int, generator: torch.Generator | None = None
) -> tuple[torch.Tensor, torch.Tensor]:
    """Draw ``noise ~ N(0, I)`` shaped like ``actions``, then one uniform timestep per row.

    Timesteps are integers in ``[0, T)`` (``T`` itself is excluded), dtype ``long``, on the
    device of ``actions``. Draw the noise first, then the timesteps, from the same generator.
    """
    noise = torch.randn(
        actions.shape, generator=generator, device=actions.device, dtype=actions.dtype
    )
    timesteps = torch.randint(
        0, num_train_timesteps, (actions.shape[0],), generator=generator, device=actions.device
    )
    return noise, timesteps

In [ ]:
# Check: Exercise 11.3
actions = torch.zeros(2000, 16, 7)
noise, t = sample_noise_and_timesteps(actions, 100, torch.Generator().manual_seed(3))
assert noise.shape == actions.shape and t.shape == (2000,) and t.dtype == torch.long, (
    "Expected noise like actions and long timesteps (B,)."
)
assert t.min() == 0 and t.max() == 99, "Timesteps must cover 0 … T-1 and never reach T."
assert abs(float(noise.mean())) < 0.01 and abs(float(noise.std()) - 1) < 0.01, (
    "Noise must be standard normal."
)
again = sample_noise_and_timesteps(actions, 100, torch.Generator().manual_seed(3))
assert torch.equal(noise, again[0]) and torch.equal(t, again[1]), (
    "The same seeded generator must give the same draws."
)
torch.manual_seed(9)
expected_noise = torch.randn(actions.shape)
expected_t = torch.randint(0, 100, (2000,))
torch.manual_seed(9)
drawn = sample_noise_and_timesteps(actions, 100)
assert torch.equal(drawn[0], expected_noise) and torch.equal(drawn[1], expected_t), (
    "Draw the noise first, then the timesteps (the pinned order)."
)
print("Exercise 11.3 passed")

# More behaviour checks: lesson11_checks.py, next to this notebook.
from lesson11_checks import check_sample_noise_and_timesteps

check_sample_noise_and_timesteps(sample_noise_and_timesteps)

## Exercise 11.4 — Masked noise loss

**Purpose.** Padded action steps are copies, not decisions. They must not train the network. This is the loss that ignores them.

**Contract (shapes, units, masks).**

- `prediction`, `target`: `(B, H, A)`. `is_pad`: `(B, H)` bool, `True` = padded step.
- Return one scalar tensor:

$$L=\frac{\sum_{b,h,a}v_{b,h}\left(\hat{\varepsilon}_{b,h,a}-\varepsilon_{b,h,a}\right)^2}{\max\!\left(A\sum_{b,h}v_{b,h},\ 1\right)}$$

  Where:
  * $\hat{\varepsilon}$ - `prediction`; $\varepsilon$ - `target`; both shape $(B,H,A)$.
  * $b$, $h$, $a$ - indices of the batch row, action step, and action value.
  * $v_{b,h}$ - 1 for a real step and 0 for padding (`~is_pad`), broadcast over $a$.
  * $A$ - action width; $A\sum v$ counts the valid scalar values in the whole batch.
- Changing `prediction` or `target` only at padded steps must change neither the loss nor the gradient. A batch that is entirely padding returns `0` (no `nan`).

**Method and source.** Mean squared error $\lVert\varepsilon-\varepsilon_\theta\rVert^2$ ([DDPM](https://arxiv.org/abs/2006.11239), equation 12 / Algorithm 1) averaged over real elements only; the pinned `compute_loss` does `(loss * mask).sum() / (mask.sum() * action_dim).clamp_min(1)`.

**Pointers.** Square the error. Multiply by a float mask. Divide by the number of valid *scalar* values, not the number of rows.

<details><summary>Detailed hint</summary>

```python
valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(prediction.dtype)        # (B, H, 1)
denominator = (valid.sum() * prediction.shape[-1]).clamp_min(1)
return ((prediction - target).square() * valid).sum() / denominator
```

</details>

*This is the reference notebook: the next cell is a complete solution.*

More behaviour checks for this exercise: `check_masked_epsilon_loss` in [lesson11_checks.py](lesson11_checks.py).

In [ ]:
# Reference solution for Exercise 11.4
def masked_epsilon_loss(
    prediction: torch.Tensor, target: torch.Tensor, is_pad: torch.Tensor
) -> torch.Tensor:
    """Mean squared error over valid ``(B, H, A)`` elements.

    ``sum((prediction - target) ** 2 * valid) / max(number_of_valid_steps * A, 1)`` with
    ``valid = ~is_pad`` broadcast over ``A``. Padded steps have zero gradient. A batch that is
    entirely padding returns zero.
    """
    valid = (~is_pad.to(torch.bool)).unsqueeze(-1).to(prediction.dtype)
    denominator = (valid.sum() * prediction.shape[-1]).clamp_min(1)
    return ((prediction - target).square() * valid).sum() / denominator

In [ ]:
# Check: Exercise 11.4
pred = torch.zeros(2, 3, 2, requires_grad=True)
target = torch.ones(2, 3, 2)
pad = torch.tensor([[False, False, True], [False, True, True]])
loss = masked_epsilon_loss(pred, target, pad)
assert loss.shape == () and torch.isfinite(loss), "Return one finite scalar."
close(loss, torch.tensor(1.0), "three valid steps, each with error 1")
loss.backward()
assert torch.count_nonzero(pred.grad[pad]) == 0, "Padded steps must have zero gradient."
close(
    pred.grad[~pad].sum(), torch.tensor(-2 * 6 / 6.0), "gradient of the valid elements", atol=1e-6
)
shaken = pred.detach().clone()
shaken[pad] = 99.0
close(
    masked_epsilon_loss(shaken, target, pad),
    loss.detach(),
    "changing padded predictions must not change the loss",
)
assert masked_epsilon_loss(pred.detach(), target, torch.ones_like(pad)) == 0, (
    "An all-padding batch gives exactly 0."
)
print("Exercise 11.4 passed")

# More behaviour checks: lesson11_checks.py, next to this notebook.
from lesson11_checks import check_masked_epsilon_loss

check_masked_epsilon_loss(masked_epsilon_loss)

## Exercise 11.5 — Join the loss operations for a model update

**Purpose.** Connect the forward process and padding mask to the callback used by a training loop.

**Contract.** The model receives `(noisy_actions, timesteps, batch)` and predicts noise with shape $(B,H,A)$. `batch` is a `dict[str, Tensor]` keyed by LeRobot feature names; it holds the clean `action` window $(B,H,A)$, its `action_is_pad` mask $(B,H)$, and the observation features. The callback returns a scalar loss and a metrics mapping `{"epsilon_mse": loss}`. Optional fixed noise and timesteps make a check repeatable. The optimizer owns `zero_grad`, `backward`, and `step`.

More behaviour checks for this exercise: `check_epsilon_loss_fn` in [lesson11_checks.py](lesson11_checks.py).

In [ ]:
def epsilon_loss_fn(
    model: DiffusionTrainingModel,
    batch: Mapping[str, torch.Tensor],
    *,
    noise: torch.Tensor | None = None,
    time: torch.Tensor | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    actions = batch["action"]
    if noise is None:
        noise = torch.randn_like(actions)
    if time is None:
        time = torch.randint(
            0, model.schedule.num_train_timesteps, (actions.shape[0],), device=actions.device
        )
    noisy = forward_noise(actions, noise, time, model.schedule.alpha_bar)
    prediction = model(noisy, time, batch)
    loss = masked_epsilon_loss(prediction, noise, batch["action_is_pad"])
    return loss, {"epsilon_mse": loss}

In [ ]:
class TinyDenoiser(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.schedule = DDPMSchedule.cosine(20)
        self.weight = nn.Parameter(torch.tensor(0.0))

    def forward(
        self, noisy: torch.Tensor, time: torch.Tensor, batch: Mapping[str, torch.Tensor]
    ) -> torch.Tensor:
        return torch.zeros_like(noisy) + self.weight


model = TinyDenoiser()
batch = {
    "action": torch.ones(3, 4, 2),
    "action_is_pad": torch.tensor(
        [[False, False, True, True], [False, False, False, True], [True, True, True, True]]
    ),
}
noise = torch.full_like(batch["action"], 0.5)
timesteps = torch.tensor([1, 7, 12])
loss, metrics = epsilon_loss_fn(model, batch, noise=noise, time=timesteps)
assert loss.shape == () and torch.allclose(loss, metrics["epsilon_mse"])
loss.backward()
assert torch.isfinite(model.weight.grad) and model.weight.grad.abs() > 0
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer.step()
assert model.weight.item() != 0.0

# More behaviour checks: lesson11_checks.py, next to this notebook.
from lesson11_checks import check_epsilon_loss_fn

check_epsilon_loss_fn(epsilon_loss_fn)

## View the noise schedule

The slider changes only the display. It does not train a model. The signal bar is $\sqrt{\bar{\alpha}_t}$ and the noise bar is $\sqrt{1-\bar{\alpha}_t}$, with $\bar{\alpha}_t=\prod_{s=0}^{t}(1-\beta_s)$. At $t=0$ the signal is near 1 and the noise is near 0. Both bars are the standard-deviation weights in the noising equation.

In [ ]:
def show_noise_level(timestep: int = 50) -> None:
    alpha_bar = (1.0 - cosine_betas(100)).cumprod(0)
    x0 = torch.ones(1, 1, 1)
    eps = torch.ones_like(x0)
    xt = forward_noise(x0, eps, torch.tensor([timestep]), alpha_bar)
    plt.bar(
        ["signal", "noise"],
        [alpha_bar[timestep].sqrt().item(), (1 - alpha_bar[timestep]).sqrt().item()],
    )
    plt.title(f"t={timestep}; noisy example={xt.item():.3f}")
    plt.show()


display(
    widgets.interactive(
        show_noise_level, timestep=widgets.IntSlider(value=0, min=0, max=99, description="t")
    )
)

## Full experiment — Train through the learner loss callback

This run uses a prepared policy and the frozen LIBERO training split. It calls the `epsilon_loss_fn` you wrote above inside the real training loop. It saves model weights, processors, statistics, and the manifest. It evaluates the held-out validation set and the same development states before and after training.

The recipe uses 2,000 updates, batch size 8, one GPU up to 48 GB, and a 100-minute training cap, which reserves up to 20 minutes for reload and evaluation within the two-hour budget. The selected run requires local weights and data. It does not start a download.

In [ ]:
CHECKPOINT = WORKSPACE / ".local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000"
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"
DEVICE = "cuda"
UPDATES = 2000
BATCH_SIZE = 8
RUN_DIR = DATA / "runs" / f"course-diffusion-training-{int(wall_clock.time())}"
BASE_POLICY, BASE_PRE, BASE_POST = load_prepared_diffusion(CHECKPOINT, device=DEVICE)
VALIDATION = open_libero_dataset(
    CONFIGS, DATA, "bowl_plate", "validation", diffusion_delta_timestamps(BASE_POLICY.config, 10)
)
VALID_ROWS = list(range(16))

In [ ]:
def validation_epsilon(
    policy: DiffusionPolicy,
    processor: Callable[[Mapping[str, torch.Tensor]], Mapping[str, torch.Tensor]],
) -> float:
    batch = processor(default_collate([VALIDATION[i] for i in VALID_ROWS]))
    device = next(policy.parameters()).device
    batch = {
        key: value.to(device) if isinstance(value, torch.Tensor) else value
        for key, value in batch.items()
    }
    policy.eval()
    torch.manual_seed(20261006)
    with torch.no_grad():
        loss, _metrics = policy(batch)
    return float(loss)


validation_before = validation_epsilon(BASE_POLICY, BASE_PRE)

In [ ]:
before = evaluate_diffusion_reference(
    checkpoint=CHECKPOINT,
    configs_root=CONFIGS,
    data_root=DATA,
    output_dir=DATA / "evaluations" / f"course-diffusion-before-{int(wall_clock.time())}",
    state_ids=[10, 11],
)

In [ ]:
assert torch.cuda.is_available(), "Full policy training needs a CUDA GPU."
assert torch.cuda.get_device_properties(0).total_memory <= 48 * 1024**3, (
    "The run target is at most 48 GB GPU memory."
)
DEN0ISER = LeRobotDenoiser(BASE_POLICY)


def learner_policy_forward(
    batch: Mapping[str, torch.Tensor],
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    return epsilon_loss_fn(DEN0ISER, batch)


BASE_POLICY.forward = learner_policy_forward


def stop_at_deadline(_signal: signal.Signals, _frame: FrameType | None) -> None:
    raise TimeoutError("Diffusion training reached its two-hour limit; keep completed checkpoints.")


signal.signal(signal.SIGALRM, stop_at_deadline)
signal.alarm(100 * 60)
try:
    TRAIN_RESULT = train_diffusion_reference(
        task_alias="bowl_plate",
        configs_root=CONFIGS,
        data_root=DATA,
        output_dir=RUN_DIR,
        steps=UPDATES,
        batch_size=BATCH_SIZE,
        checkpoint_every=500,
        device=DEVICE,
        policy=BASE_POLICY,
    )
finally:
    signal.alarm(0)

In [ ]:
FINAL_CHECKPOINT = TRAIN_RESULT.output_dir / f"checkpoint-{UPDATES:06d}"
RELOADED, RELOADED_PRE, RELOADED_POST, TRAINED_MANIFEST = load_diffusion_bundle(FINAL_CHECKPOINT)
validation_after = validation_epsilon(RELOADED, RELOADED_PRE)
after = evaluate_diffusion_reference(
    checkpoint=FINAL_CHECKPOINT,
    configs_root=CONFIGS,
    data_root=DATA,
    output_dir=DATA / "evaluations" / f"course-diffusion-after-{int(wall_clock.time())}",
    state_ids=[10, 11],
)
print(
    {
        "updates": TRAIN_RESULT.steps,
        "last_train_loss": TRAIN_RESULT.last_loss,
        "validation_before": validation_before,
        "validation_after": validation_after,
        "success_before": before["success_rate"],
        "success_after": after["success_rate"],
        "peak_gpu_mib": torch.cuda.max_memory_allocated() / 1024**2,
    }
)